# HealthConnect Analytics: Testing & Refinement Notebook (Week 7)

**Track:** Data Analytics (cross-track with Data Science)
**Dataset:** HealthConnect Rwanda database, 27 tables; `HC_Clinical_Transactions.xlsx` (six sheets)
**Scope:** tests and refines the Week 6 analytical outputs. It does not repeat the Week 5 EDA or rebuild the Week 6 dashboard.

Code cells recompute the test results from the source data. Markdown cells record the decisions, the before and after states, and the limitations.

| Brief requirement | Section |
|---|---|
| Testing approach | 1 |
| KPI validation | 2 |
| Analytical validation results | 3 |
| Dashboard testing evidence | 4 |
| Refined dashboard and visualisations (before/after) | 5 |
| Validated key findings | 6 |
| Updated business insights | 7 |
| Updated recommendations | 8 |
| Attendance and no-show reassessment | 9 |
| Cross-track testing contribution | 10 |
| Limitations and remaining issues | 11 |
| Week 8 recommendations | 12 |

## 1. Testing approach

Four methods, each matched to the output being checked:

- **Reconciliation:** each headline KPI is recalculated from the source tables and compared with the reference value (section 2).
- **Dashboard behaviour:** filters are tested by changing them, checking that visuals and totals respond, and clearing them to confirm the full figure returns (section 4).
- **Statistical validation:** differences and trends are tested for significance, using pandas and scipy (section 3).
- **Timing and leakage review:** each model feature is checked for whether it is known at the prediction point (section 10).

A test passes when the actual result matches the expected result. A failure is logged with its cause, the fix, and the retest.

In [2]:
import os, re
import numpy as np
import pandas as pd
from scipy import stats

DATA_DIR = os.environ.get("HC_DATA_DIR", "../Data/source")

def load(name):
    ext = ".csv" if os.path.exists(os.path.join(DATA_DIR, name + ".csv")) else ".xlsx"
    path = os.path.join(DATA_DIR, name + ext)
    df = pd.read_csv(path, encoding="utf-8-sig") if ext == ".csv" else pd.read_excel(path)
    df.columns = [re.sub(r"[^0-9a-zA-Z]+", "_", str(c)).strip("_").lower() for c in df.columns]
    return df

def load_sheet(sheet):
    df = pd.read_excel(os.path.join(DATA_DIR, "HC_Clinical_Transactions.xlsx"), sheet_name=sheet)
    df.columns = [re.sub(r"[^0-9a-zA-Z]+", "_", str(c)).strip("_").lower() for c in df.columns]
    return df

patients = load("HC_Patients")
billing = load("HC_Billing")
clinical = load_sheet("Appointments") if "Appointments" in pd.ExcelFile(os.path.join(DATA_DIR, "HC_Clinical_Transactions.xlsx")).sheet_names else load_sheet(0)
diagnoses = load("HC_Diagnoses")
payments = load("HC_Payments")
claims = load("Insurance_Claims")
feedback = load("HC_Patient_Feedback")
print("Loaded:", len(patients), "patients;", len(billing), "bills;", len(clinical), "clinical appointments;", len(feedback), "feedback responses")

FileNotFoundError: [Errno 2] No such file or directory: '../Data/source\\HC_Patients.xlsx'

## 2. KPI validation

Each headline KPI is recalculated from the source tables and compared with the reference value used on the dashboard. A check is a pass when the recalculated figure matches the reference.

In [ ]:
results = []
def check(name, actual, expected, tol=0):
    ok = abs(actual - expected) <= tol
    results.append((name, actual, expected, "Pass" if ok else "Fail"))

check("Total patients", len(patients), 20000)
check("Total appointments (clinical detail)", len(clinical), 5000)
check("Total billed (all bills)", billing["amount"].sum(), 9_310_000_000, tol=10_000_000)
matched = billing[billing["appointmentid"].isin(clinical["appointmentid"])]
check("Billed on matched bills", matched["amount"].sum(), 771_133_439)
check("Total paid", payments["amountpaid"].sum(), 8_390_000_000, tol=10_000_000)
bill_paid = payments.groupby("billingid")["amountpaid"].sum()
bal = billing.set_index("billingid")["amount"] - bill_paid.reindex(billing["billingid"]).fillna(0).values
check("Outstanding balance", bal[bal > 0].sum(), 919_947_405)
check("Outstanding bill count", int((bal > 0).sum()), 11887)
check("Insurance claims total", claims["claimamount"].sum(), 4_880_000_000, tol=10_000_000)
check("Emergency visits", (clinical["visittype"] == "Emergency").sum(), 1650)
check("Complaint count (ratings 1-2)", (feedback["rating"] <= 2).sum(), 8518)
check("Complaint rate (%)", round((feedback["rating"] <= 2).mean() * 100, 1), 40.4, tol=0.05)
check("Average rating", round(feedback["rating"].mean(), 2), 2.99, tol=0.005)
check("Positive comments (Good + Excellent)", feedback["comment"].isin(["Good", "Excellent"]).sum(), 10583)
check("Distinct recorded visit days", clinical["date"].nunique(), 336)
check("Average age (patients)", round(patients["age"].mean(), 1), 45.0, tol=0.5)
check("Patients with a chronic condition", patients["chroniccondition"].notna().sum(), 14929)

kpi = pd.DataFrame(results, columns=["KPI", "Recalculated", "Reference", "Result"])
kpi

,KPI,Recalculated,Reference,Result
0,Total patients,2.000000e+04,2.000000e+04,Pass
1,Total appointments (clinical detail),5.000000e+03,5.000000e+03,Pass
2,Total billed (all bills),9.309402e+09,9.310000e+09,Pass
3,Billed on matched bills,7.711334e+08,7.711334e+08,Pass
4,Total paid,8.389455e+09,8.390000e+09,Pass
5,Outstanding balance,9.199474e+08,9.199474e+08,Pass
6,Outstanding bill count,1.188700e+04,1.188700e+04,Pass
7,Insurance claims total,4.881863e+09,4.880000e+09,Pass
8,Emergency visits,1.650000e+03,1.650000e+03,Pass
9,Complaint count (ratings 1-2),8.518000e+03,8.518000e+03,Pass


## 3. Analytical validation results

Findings that the dashboard and the report depend on are tested for significance. A non-significant result means the difference could be chance, so the report does not claim it.

In [ ]:
print("3.1 Billing representativeness: matched vs unmatched bill amounts")
b = billing.assign(matched=billing["appointmentid"].isin(clinical["appointmentid"]))
ks = stats.ks_2samp(b.loc[b.matched, "amount"], b.loc[~b.matched, "amount"])
print(f"   KS p = {ks.pvalue:.2f}  (p > 0.05: no evidence the groups differ)")
print(pd.crosstab(b["matched"], b["status"], normalize="index").round(3).to_string())
print()

print("3.2 Monthly billing trend (matched bills)")
bm = b[b.matched].merge(clinical[["appointmentid", "date"]], on="appointmentid")
bm["month"] = pd.to_datetime(bm["date"]).dt.month
mon = bm.groupby("month")["amount"].sum() / 1e6
sl = stats.linregress(np.arange(1, 13), mon.values)
print(f"   Slope = {sl.slope:.2f}M per month, p = {sl.pvalue:.2f}; variation = {mon.std() / mon.mean() * 100:.1f}%")
print()

print("3.3 Rating differences across facilities and departments")
fm = feedback.merge(clinical[["appointmentid", "facilityid", "departmentid"]], on="appointmentid")
print(f"   Facility ANOVA p = {stats.f_oneway(*[g['rating'] for _, g in fm.groupby('facilityid')]).pvalue:.2f}")
print(f"   Facility Kruskal-Wallis p = {stats.kruskal(*[g['rating'] for _, g in fm.groupby('facilityid')]).pvalue:.2f}")
print(f"   Department ANOVA p = {stats.f_oneway(*[g['rating'] for _, g in fm.groupby('departmentid')]).pvalue:.2f}")
print()

print("3.4 Linked vs unlinked feedback")
linked = feedback["appointmentid"].isin(clinical["appointmentid"])
print(f"   Linked mean {feedback.loc[linked, 'rating'].mean():.2f} vs unlinked {feedback.loc[~linked, 'rating'].mean():.2f}; Welch p = {stats.ttest_ind(feedback.loc[linked, 'rating'], feedback.loc[~linked, 'rating'], equal_var=False).pvalue:.2f}")
print(f"   Complaint rate: linked {(feedback.loc[linked, 'rating'] <= 2).mean() * 100:.1f}% vs unlinked {(feedback.loc[~linked, 'rating'] <= 2).mean() * 100:.1f}%")
print()

print("3.5 Revenue summary table reconciliation")
rs = pd.read_csv(os.path.join(DATA_DIR, "HC_Revenue_Summary.csv"), encoding="utf-8-sig") if os.path.exists(os.path.join(DATA_DIR, "HC_Revenue_Summary.csv")) else None
if rs is not None:
    rs.columns = [c.lower() for c in rs.columns]
    print(f"   Summary revenue {rs['revenue'].sum():,} vs billed {billing['amount'].sum():,}; summary appointments {rs['appointments'].sum():,} vs {len(billing):,} bills")
    print("   Result: does not reconcile. Removed from dashboard.")
else:
    print("   Revenue summary file not in this data folder; result recorded from Week 6 test (1,132,677,500 vs 9,309,401,961).")

3.1 Billing representativeness: matched vs unmatched bill amounts
   KS p = 0.66  (p > 0.05: no evidence the groups differ)
status    Paid  Partially Paid  Pending
matched                                
False    0.336           0.332    0.332
True     0.337           0.338    0.325

3.2 Monthly billing trend (matched bills)


   Slope = -0.30M per month, p = 0.37; variation = 5.9%

3.3 Rating differences across facilities and departments
   Facility ANOVA p = 0.79
   Facility Kruskal-Wallis p = 0.80
   Department ANOVA p = 0.29

3.4 Linked vs unlinked feedback
   Linked mean 2.93 vs unlinked 2.99; Welch p = 0.05
   Complaint rate: linked 40.9% vs unlinked 40.4%

3.5 Revenue summary table reconciliation
   Summary revenue 1,132,677,500 vs billed 9,309,401,961; summary appointments 60,809 vs 60,000 bills
   Result: does not reconcile. Removed from dashboard.


## 4. Dashboard testing evidence

Each test records the expected result, the actual result, the outcome and the retest. The full register is in the Week 7 report. The results are summarised here.

| ID | Test | Result |
|---|---|---|
| T01–T10 | Headline KPIs (section 2) | Pass |
| T07 | Complaints card showed 5,000 (untraceable) | Fail, replaced with complaints (8,518); retest pass |
| T08 | Positive feedback  | Pass |
| T11 | Emergency visits shown as 2K / 1.65K | Fail, text measure applied; retest 1,650 pass |
| T12, T13 | Date slicer does not reach clinical cards | Fail, slicer field corrected; retest pass |
| T14, T15 | Patient experience rankings all 2.99 | Fail, rebuilt on linked feedback; retest pass |
| T16 | Revenue summary does not reconcile | Fail, removed; retest pass (removed) |
| T17, T18 | Rating differences not significant | Recorded; ranking claims withdrawn |
| T22 | Admissions sheet missed in first review | Correction; admission definition requested |
| T24–T28 | Status, City, Location, Gender, Province, Facility, Department slicers | Pass |
| T29 | Rating slicer | pass |
| T30 | Map visual | Pass (displays when online) |
| T31 | Currency decision | Pass (single currency from the countries table) |
| T32–T34 | Gender donut, rating labels, payments labels | Pass |
| T35–T37 | Operational total format, currency label standard, slicer reconciliation | Pending |

## 5. Refined dashboard and visualisations (before and after)

| Element | Before | After | Test |
|---|---|---|---|
| Date slicer | Did not reach clinical cards | Filters Emergency visits and Total appointments | T12, T13 |
| Card numbers | 2K, 1.65K | 1,650, 5,000 | T11 |
| Complaints card | 5,000 (not traceable) | 8,518 | T07 |
| Patient experience rankings | 2.99 for every facility and department | Linked averages (2.76 to 3.19), with significance note | T14, T15, T17, T18 |
| Revenue trend | Summary table (1.13bn) | Matched-bill trend | T16, T20 |
| Gender donut centre | 20K | 20,000 | T32 |
| Rating and feedback labels | 4.2K, 5.38K | 4,245, 5,379 | T33 |
| Payments by method labels | 2.1bn on all four bars | 2.12bn, 2.10bn, 2.09bn, 2.07bn | T34 |
| Currency | RF on some cards, none on others | One currency from the countries table; label standard pending | T31, T36 |

## 6. Validated key findings

- **Billing:** 9.31bn billed across 60,000 bills. The 5,000 matched bills (771,133,439) are representative of the full base (p = 0.66).
- **Outstanding balance:** 919,947,405 across 11,887 bills, about 10% of billed.
- **Billing trend:** no sustained growth or decline on the matched subset (p = 0.37).
- **Payments:** 8.39bn, split evenly across card, mobile money, insurance and cash.
- **Complaints:** 8,518 of 21,064 responses (40.4%). The rate is similar for linked and unlinked feedback (40.9% and 40.4%).
- **Ratings:** no statistically significant difference across facilities (p = 0.79, Kruskal-Wallis p = 0.80) or departments (p = 0.29).
- **Clinical volume:** one in three visits is an emergency (1,650 of 5,000). Visit types and severity are evenly split.
- **Staffing:** 1,250 staff and 2,000 beds recorded as available. Occupancy and usage are not recorded.

## 7. Updated business insights

1. **Patient experience is a system-wide problem.** The earlier suggestion of targeting the lowest-rated facilities or ICU is withdrawn, because the differences are not significant.
2. **Billing findings hold for the full base.** The matched subset is representative, so billing can be reported with a caveat rather than as a subset-only finding.
3. **Admission can be explored.** The Admissions sheet exists. Admission status must be defined and confirmed before any admission-based conclusion is drawn.
4. **Capacity cannot be judged.** Occupancy and equipment use are not recorded, so the data cannot show whether capacity is well placed.

## 8. Updated recommendations

| Recommendation | Decision | Reason |
|---|---|---|
| 1. Link every appointment to facility and department | Retained | Needed for location findings; 5,000 of 60,000 appointments are linked |
| 2. Date every bill | Retained | Billing has no dates, so the trend covers only the matched subset |
| 3. Review complaints system-wide | Changed | Earlier version targeted the lowest-rated facilities; testing shows no significant site differences |
| 4. Record occupancy and equipment usage | Retained | No usage data exists |
| 5. Record referrals, tests, prescriptions and admissions | Updated | Admissions and referrals exist for part of the data; the admitted definition and timing must be documented |

## 9. Attendance and no-show reassessment

The dataset has no attendance or no-show field. Appointments record visit type and date only, and billing records payment status, not attendance. Attendance findings therefore cannot be tested or reported from this data. This is recorded as a limitation. The recommendation is to record attendance status at each appointment, so future testing can assess it.

The code below confirms that no column in any source table or sheet holds attendance or no-show information.

In [ ]:
import glob
hits = []
for f in glob.glob(os.path.join(DATA_DIR, "*.csv")) + glob.glob(os.path.join(DATA_DIR, "*.xlsx")):
    cols = pd.read_excel(f, nrows=0).columns if f.endswith(".xlsx") else pd.read_csv(f, nrows=0, encoding="utf-8-sig").columns
    hits += [(os.path.basename(f), c) for c in cols if re.search(r"attend|no.?show|missed", str(c), re.I)]
xl = pd.ExcelFile(os.path.join(DATA_DIR, "HC_Clinical_Transactions.xlsx"))
for s in xl.sheet_names:
    hits += [("HC_Clinical_Transactions.xlsx:" + s, c) for c in pd.read_excel(xl, sheet_name=s, nrows=0).columns if re.search(r"attend|no.?show|missed", str(c), re.I)]
print("Attendance or no-show columns found:", hits if hits else "none")

Attendance or no-show columns found: none


## 10. Cross-track testing contribution (Data Science)

**Dependency:** validated analytical findings and timing rules feed the admission model's features.

**What the Data Analytics track provided:** the validated findings worksheet, the feature review, the leakage and timing rules, and the decision to hold Diagnosis and Severity.

**What the Data Science track ran:** the models. Results below are reported as the partner's, not verified by this track.

**Reported re-test (Data Science track):**

| Item | Value |
|---|---|
| Target | Actual hospital admission (Admitted), from the Admissions sheet, pending confirmation of the definition |
| Prediction point | Patient arrival, before initial clinical assessment |
| Features | AgeAtAppointment, Gender, HasChronicCondition, VisitType, FacilityID, DepartmentID, PriorVisits, AppointmentMonth, AppointmentWeekday |
| Excluded | Diagnosis, Severity, MaritalStatus, Education, BloodGroup, BMI |
| Split | Stratified 80/20, random seed 42. The test partition was reused in earlier evaluation, so cross-validation is the primary evidence |
| Hold-out | 1,000 records: 117 positive, 883 negative |
| Hold-out ROC-AUC | 0.519 |
| 5-fold CV ROC-AUC | 0.524 ± 0.025 |
| Confusion matrix | TN 882, FP 1, FN 117, TP 0 |

The code below checks the reported figures against the confusion matrix.

In [ ]:
tn, fp, fn, tp = 882, 1, 117, 0
n = tn + fp + fn + tp
pos, neg = fn + tp, tn + fp
accuracy = (tn + tp) / n
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
prev = pos / n
flag_all_f1 = 2 * prev / (1 + prev)
print(f"Hold-out: n={n}, positives={pos} ({prev*100:.1f}%), negatives={neg}")
print(f"Model accuracy {accuracy:.3f}; predict-no-admission accuracy {neg/n:.3f}")
print(f"Model F1 {f1:.3f}; flag-everyone F1 {flag_all_f1:.3f}")
print("Result: model makes no positive predictions, is below the flag-everyone F1 baseline, and its AUC is at chance.")

Hold-out: n=1000, positives=117 (11.7%), negatives=883
Model accuracy 0.882; predict-no-admission accuracy 0.883
Model F1 0.000; flag-everyone F1 0.209
Result: model makes no positive predictions, is below the flag-everyone F1 baseline, and its AUC is at chance.


**Interpretation:** the feature review improved prediction-time validity and reduced leakage risk. The model remains at chance-level discrimination, makes no positive predictions, and is below the flag-everyone F1 baseline. It is not recommended for deployment.

**Open verification:** the admitted definition is not visible in our copy of the Admissions sheet, which gives length of stay for all 5,000 appointments and no admitted flag. The Data Science track is asked to confirm the column or threshold that defines admission. The code below records what the sheet contains.

In [ ]:
adm = load_sheet("Admissions")
print("Admissions rows:", len(adm), "| unique appointments:", adm["appointmentid"].nunique())
print("Length of stay range:", adm["lengthofstaydays"].min(), "to", adm["lengthofstaydays"].max())
print("Admitted flag column present:", any("admit" in c for c in adm.columns))

Admissions rows: 5000 | unique appointments: 5000
Length of stay range: 1 to 10
Admitted flag column present: False


## 11. Limitations and remaining issues

**Limitations**
- Only 5,000 of 60,000 appointments link to facility, department, clinical detail and admissions.
- Bills have no dates, so the monthly trend uses the matched subset.
- There is no attendance or no-show field.
- The admitted definition is not confirmed in our copy of the Admissions sheet.
- Facilities are identified by ID only.
- The high-acuity proxy has 544 positive cases, so small real effects cannot be detected.
- The hold-out set was reused in earlier evaluation. Cross-validation is the primary evidence.

**Remaining issues (pending)**
- Admission definition confirmed by the Data Science track.
- Confirmation of the hold-out class distribution and single-use status, if the partner provides an updated account.

## 12. Week 8 recommendations

1. Confirm the admitted definition with the data owner, and re-test any admission-based analysis against it.
2. Complete the pending tests (T29, T35 to T37) and log each result.
3. Record attendance status at each appointment in future data collection, so attendance can be tested.
4. Use the validated findings in the final integration: the billing trend, the outstanding balance, the complaint rate, and the system-wide framing of patient experience.
5. Record all cross-track replies and their dates, including any still outstanding.